In [1]:
import polars as pl
from pathlib import Path

pdb_df = pl.read_parquet("../../data/pdb/triad/staged/pdb_triad.neg.parquet")
af3_inf_dir = Path("../../data/pdb/triad/inference")

In [2]:
from mdaf3.AF3OutputParser import AF3Output
from mdaf3.FeatureExtraction import *
import numpy as np
import polars as pl
from scipy.stats import gmean
import warnings

warnings.filterwarnings("ignore")


def extract_mean_tcr_pmhc_pae(row, af3_parent_dir, **kwargs):
    af3_output = AF3Output(af3_parent_dir / row["job_name"], **kwargs)
    u = af3_output.get_mda_universe(**kwargs)

    peptide_res = u.select_atoms("segid A").residues

    if row["mhc_class"] == "II":
        mhc_residx = u.select_atoms("segid B or segid C").residues.resindices
    else:
        mhc_residx = u.select_atoms("segid B").residues.resindices

    tcr_residx = u.select_atoms("segid D or segid E").residues.resindices

    pae = af3_output.get_pae_ndarr(**kwargs)

    if row["mhc_class"] == "II":
        if len(peptide_res) <= 9:
            row["mean_p_tcr_pae"] = (
                np.mean(pae[peptide_res.resindices][:, tcr_residx]) / 100
            )
            row["mean_tcr_p_pae"] = (
                np.mean(pae[tcr_residx][:, peptide_res.resindices]) / 100
            )
        else:

            p_tcr_window_means = []
            tcr_p_window_means = []
            for i in range(len(peptide_res) - 8):
                p_tcr_window_means.append(
                    (
                        np.mean(pae[peptide_res[i : i + 9].resindices][:, tcr_residx])
                        / 100
                    )
                )
                tcr_p_window_means.append(
                    (
                        np.mean(pae[tcr_residx][:, peptide_res[i : i + 9].resindices])
                        / 100
                    )
                )
            row["mean_p_tcr_pae"] = gmean(p_tcr_window_means)
            row["mean_tcr_p_pae"] = gmean(tcr_p_window_means)

    else:
        row["mean_p_tcr_pae"] = (
            np.mean(pae[peptide_res.resindices][:, tcr_residx]) / 100
        )
        row["mean_tcr_p_pae"] = (
            np.mean(pae[tcr_residx][:, peptide_res.resindices]) / 100
        )

    # mean_interface_pae = np.mean(pae[pmhc_residx][:, tcr_residx])

    row["mean_mhc_tcr_pae"] = np.mean(pae[mhc_residx][:, tcr_residx])
    row["mean_tcr_mhc_pae"] = np.mean(pae[tcr_residx][:, mhc_residx])

    return row


pdb_df = split_apply_combine(
    pdb_df, extract_mean_tcr_pmhc_pae, af3_inf_dir, chunksize=15
)

/home/lwoods/miniconda3/envs/tcrtrifold-experiments/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1672/1672 [00:08<00:00, 195.28it/s]


In [4]:
from sklearn.metrics import roc_auc_score


np_dat = pdb_df.select("mean_p_tcr_pae", "cognate").to_numpy()

y_score = 1 - np_dat[:, 0]
y_true = np_dat[:, 1]

roc_auc_score(y_true, y_score)

0.8466153047091413